# V6.7 — YOLO26m cumulative ablation: M1, M2, M3

Train identical YOLO26m configurations on `m1_a_b_f`, `m2_c_d`, and `m3_e_g`. Each stage has independent checkpoints and a Camber mirror under `stash://bunpmc/projects/finding_dori/`. Nothing trains until a stage cell is run.

## 1. Setup

In [1]:
from pathlib import Path
import json, os, subprocess, time
import yaml
def find_repo(start):
    for p in (start.resolve(), *start.resolve().parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir(): return p
    raise FileNotFoundError('Repository root not found')
REPO = find_repo(Path.cwd())
BENCHMARK = REPO / 'benchmakring_transmission_line_defect_model'
DATA_ROOT = BENCHMARK / 'data/curation_module_ablation'
RUN_ROOT = BENCHMARK / 'runs/curation_module_ablation'
REMOTE_BASE = 'stash://bunpmc/projects/finding_dori'
STAGES = {'m1_a_b_f': {'modules':[1], 'images':18334}, 'm2_c_d': {'modules':[1,2], 'images':18052}, 'm3_e_g': {'modules':[1,2,3], 'images':17960}}
candidates = (BENCHMARK/'weights/yolo26m.pt', BENCHMARK/'yolo26m.pt', REPO/'yolo26m.pt')
WEIGHTS = next((p for p in candidates if p.is_file()), Path('yolo26m.pt'))
print('Repository:', REPO)
print('Weights:', WEIGHTS)
print('Stages:', list(STAGES))

Repository: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection
Weights: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\yolo26m.pt
Stages: ['m1_a_b_f', 'm2_c_d', 'm3_e_g']


## 2. Preflight all datasets (read-only)

In [2]:
for stage, expected in STAGES.items():
    root = DATA_ROOT / stage
    cfg = yaml.safe_load((root/'data.yaml').read_text(encoding='utf-8'))
    manifest = json.loads((root/'manifest.json').read_text(encoding='utf-8'))
    assert manifest['modules'] == expected['modules']
    assert cfg['val'] == 'images/val' and cfg['test'] == 'images/val_clean' and len(cfg['names']) == 6
    counts = {}
    for split in ('train','val','val_clean'):
        ni = len([p for p in (root/'images'/split).iterdir() if p.is_file() and p.suffix != '.cache'])
        nl = len(list((root/'labels'/split).glob('*.txt')))
        assert ni == nl, (stage, split, ni, nl)
        counts[split] = ni
    assert counts == {'train':expected['images'], 'val':2272, 'val_clean':2271}
    print(stage, expected['modules'], counts)

m1_a_b_f [1] {'train': 18334, 'val': 2272, 'val_clean': 2271}
m2_c_d [1, 2] {'train': 18052, 'val': 2272, 'val_clean': 2271}
m3_e_g [1, 2, 3] {'train': 17960, 'val': 2272, 'val_clean': 2271}


## 3. Training, resume, and Camber helpers

Syncs every 5 epochs, on interruption, and after completion.

In [3]:
from ultralytics import YOLO
def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'): return
    for line in (REPO/'.env').read_text(encoding='utf-8').splitlines():
        key, sep, value = line.partition('=')
        if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
            os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('CAMBER_API_KEY missing from .env')
def wsl_path(path):
    path = Path(path).resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'
def camber_cp(local, remote, attempts=3):
    load_camber_key(); env = os.environ.copy()
    parts = [x for x in env.get('WSLENV','').split(':') if x]
    if 'CAMBER_API_KEY' not in parts: parts.append('CAMBER_API_KEY')
    env['WSLENV'] = ':'.join(parts)
    cmd = ['wsl','--','/home/dotru/.camber/bin/camber','stash','cp',wsl_path(local),remote]
    for attempt in range(1, attempts+1):
        proc = subprocess.run(cmd,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        if proc.returncode == 0: return
        if attempt < attempts: time.sleep(5*attempt)
    raise RuntimeError(proc.stdout[-2000:])
def paths(stage):
    return DATA_ROOT/stage, RUN_ROOT/stage/'yolo26m', f'{REMOTE_BASE}/yolo26m_{stage}'
def epochs_done(run_dir):
    p = run_dir/'results.csv'
    return max(0,len(p.read_text(encoding='utf-8').splitlines())-1) if p.is_file() else 0
def sync_run(run_dir, remote, trainer=None, final=False):
    files = [run_dir/'weights/last.pt',run_dir/'weights/best.pt',run_dir/'args.yaml',run_dir/'results.csv']
    if trainer is not None:
        p = run_dir/f'weights/epoch{trainer.epoch+1}.pt'
        if p.is_file(): files.append(p)
    if final: files = [p for p in run_dir.rglob('*') if p.is_file() and p.suffix != '.cache']
    for p in files:
        if p.is_file(): camber_cp(p,f'{remote}/{p.relative_to(run_dir).as_posix()}')
def train_stage(stage):
    if stage not in STAGES: raise KeyError(stage)
    data_root, run_dir, remote = paths(stage)
    data, manifest = data_root/'data.yaml', data_root/'manifest.json'
    camber_cp(data,f'{remote}/data.yaml'); camber_cp(manifest,f'{remote}/dataset_manifest.json')
    last_pt, best_pt = run_dir/'weights/last.pt', run_dir/'weights/best.pt'
    if epochs_done(run_dir) >= 50 and best_pt.is_file():
        print(f'[SKIP] {stage} completed'); sync_run(run_dir,remote,final=True); return None
    resume = last_pt.is_file(); model = YOLO(str(last_pt if resume else WEIGHTS))
    def on_save(trainer):
        epoch = trainer.epoch+1
        if epoch%5 == 0 or epoch == trainer.epochs:
            try: sync_run(run_dir,remote,trainer=trainer)
            except Exception as exc: print('Camber sync warning:',exc,flush=True)
    model.add_callback('on_model_save',on_save)
    print(f'[{stage}] {"RESUME" if resume else "START"} -> {run_dir}')
    try:
        if resume: result = model.train(resume=True)
        else:
            result = model.train(data=str(data),project=str(RUN_ROOT/stage),name='yolo26m',exist_ok=True,epochs=50,imgsz=640,batch=8,device=0,workers=2,optimizer='AdamW',lr0=0.001,lrf=0.01,cos_lr=True,warmup_epochs=3.0,weight_decay=0.0005,close_mosaic=10,amp=True,seed=42,deterministic=True,pretrained=True,val=True,save=True,save_period=5,plots=True,cache=False)
    except BaseException:
        sync_run(run_dir,remote); raise
    sync_run(run_dir,remote,final=True)
    print(f'[{stage}] complete -> {remote}'); return result

## 4. Train M1 — Module I (A, B, F)

In [4]:
m1_result = train_stage('m1_a_b_f')

[m1_a_b_f] START -> D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\runs\curation_module_ablation\m1_a_b_f\yolo26m
New https://pypi.org/project/ultralytics/8.4.202 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\curation_module_ablation\m1_a_b_f\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0,

## 5. Train M2 — Modules I + II

In [5]:
m2_result = train_stage('m2_c_d')

[m2_c_d] START -> D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\runs\curation_module_ablation\m2_c_d\yolo26m
New https://pypi.org/project/ultralytics/8.4.202 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\curation_module_ablation\m2_c_d\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=

## 6. Train M3 — Modules I + II + III

In [6]:
m3_result = train_stage('m3_e_g')

[SKIP] m3_e_g completed


## 7. Status

In [7]:
for stage in STAGES:
    _, run_dir, remote = paths(stage)
    print(f'{stage:10} epochs={epochs_done(run_dir):2}/50 last={(run_dir/"weights/last.pt").is_file()} best={(run_dir/"weights/best.pt").is_file()} {remote}')

m1_a_b_f   epochs=50/50 last=True best=True stash://bunpmc/projects/finding_dori/yolo26m_m1_a_b_f
m2_c_d     epochs=50/50 last=True best=True stash://bunpmc/projects/finding_dori/yolo26m_m2_c_d
m3_e_g     epochs=50/50 last=True best=True stash://bunpmc/projects/finding_dori/yolo26m_m3_e_g


## 8. Held-out test evaluation

Run after training. Evaluates every available `best.pt` on frozen `test = val_clean`.

In [8]:
test_results = {}
for stage in STAGES:
    data_root, run_dir, _ = paths(stage); best = run_dir/'weights/best.pt'
    if not best.is_file(): print(f'[SKIP TEST] {stage}: best.pt missing'); continue
    test_results[stage] = YOLO(str(best)).val(data=str(data_root/'data.yaml'),split='test',imgsz=640,batch=8,device=0,workers=2,project=str(run_dir),name='test',plots=True)

Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26m summary (fused): 130 layers, 20,354,078 parameters, 0 gradients, 68.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.30.0 ms, read: 17.010.9 MB/s, size: 118.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\curation_module_ablation\m1_a_b_f\labels\val_clean... 2271 images, 201 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2271/2271 614.7it/s 3.7s0.1s
val: New cache created: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\curation_module_ablation\m1_a_b_f\labels\val_clean.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-